In [1]:
import os
import requests
from bs4 import BeautifulSoup 
from typing import List
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic

In [2]:
import gradio as gr
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

In [3]:
# Initialize the OpenAI-compatible client for Gemini
gemini_via_openai_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [4]:
Llama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
MODEL = "llama3.2"

def llama(prompt, mood):
    messages = [
        {"role": "system", "content": mood},
        {"role": "user", "content": prompt}
      ]
    response = Llama.chat.completions.create(
    model=MODEL,
    messages=messages,
)
    return response.choices[0].message.content

In [5]:
def gemini(prompt, mood):
    messages = [
        {"role": "system", "content": mood},
        {"role": "user", "content": prompt}
      ]
    response = gemini_via_openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
)
    return response.choices[0].message.content

In [18]:
def llm_model(prompt, mood_instruction, model_type):
    if model_type == "Gemini":
        results = gemini(prompt, mood_instruction)
    elif model_type == "Llama3.2":
        results = llama(prompt, mood_instruction)
    else:
        results = "Unknown model"
    return results

# Handles the mood selection and passes it to the LLM
def mood_model(mood_type):
    if mood_type == "Funny response":
        mood_instruction = "You respond in a funny way."
    elif mood_type == "Angry response":
        mood_instruction = "You respond in a very angry manner."
    else:
        mood_instruction = "You respond normally."
    return mood_instruction

def combine(prompt, mood_type, model_type):
    return llm_model(prompt,mood_model(mood_type),model_type)

In [20]:
view = gr.Interface(fn=combine, 
                    inputs=[gr.Textbox(label="Your message:"), 
                            gr.Dropdown(["Funny response", "Angry response"], label="Select mood", value="Funny response"),
                            gr.Dropdown(["Gemini", "Llama3.2"], label="Select model", value="Gemini")],
                    outputs=[gr.Textbox(label="Response:")],
                    flagging_mode="never",            
        )

view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://317dcc736bd752e7f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
